<a href="https://colab.research.google.com/github/lamaljalal/NLP/blob/main/Group9_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Group 9 — SMS Spam Detection: Performance–Efficiency Trade-off

**Approved primary question:** Do the classification performance gains of fine-tuned DistilBERT justify its additional training and inference costs relative to frozen MiniLM embeddings and TF-IDF?

**Protocol:** fixed stratified 70/15/15 splits; tune on development only; evaluate held-out test once; full-pipeline timing from raw SMS text; report hardware, repetitions, batch size and tuning costs. Natural text-speak is a secondary exploratory analysis. Comparing different encoder architectures does **not** isolate the effect of fine-tuning.

## Setup

In [1]:
%pip install -q requests scikit-learn sentence-transformers "transformers>=4.44,<5" datasets accelerate torch pandas joblib

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [2]:
import os, sys, time, platform, json, random, statistics, importlib.metadata as im
from pathlib import Path
import numpy as np, pandas as pd, torch
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, recall_score, classification_report, confusion_matrix
from IPython.display import display
SEED=42
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
VERSIONS={p:im.version(p) for p in ['torch','scikit-learn','transformers','sentence-transformers','datasets','numpy']}
HARDWARE={'platform':platform.platform(),'python':sys.version.split()[0], 'cpu':platform.processor() or 'not reported', 'logical_cpu_count':os.cpu_count(), 'device':DEVICE, 'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,'versions':VERSIONS}
print(json.dumps(HARDWARE,indent=2))

{
  "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
  "python": "3.13.18",
  "cpu": "x86_64",
  "logical_cpu_count": 2,
  "device": "cpu",
  "gpu": null,
  "versions": {
    "torch": "2.14.1+cpu",
    "scikit-learn": "1.6.1",
    "transformers": "4.57.6",
    "sentence-transformers": "5.7.0",
    "datasets": "4.8.5",
    "numpy": "2.4.6"
  }
}


## Data: public UCI SMS Spam Collection, exact deduplication, stratified splits

In [3]:

import pandas as pd
import zipfile
import io
import requests
from sklearn.model_selection import train_test_split

# Download the official UCI SMS Spam Collection
url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"

response = requests.get(url, timeout=60)
response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    with z.open("SMSSpamCollection") as f:
        df = pd.read_csv(
            f,
            sep="\t",
            header=None,
            names=["label", "text"],
            encoding="utf-8"
        )

# Clean data
df = df.dropna(subset=["label", "text"]).copy()
df["text"] = df["text"].astype(str).str.strip()
df["label"] = df["label"].astype(str).str.strip().str.lower()
df = df[df["label"].isin(["ham", "spam"])]
df = df[df["text"].ne("")]
df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)

# Train / Development / Test split (70/15/15)
train, tmp = train_test_split(
    df, test_size=0.30,
    stratify=df["label"], random_state=SEED
)

dev, test = train_test_split(
    tmp, test_size=0.50,
    stratify=tmp["label"], random_state=SEED
)

assert not (set(train.text) & set(test.text))
assert not (set(train.text) & set(dev.text))
assert not (set(dev.text) & set(test.text))

Xtr, Xdev, Xte = (
    train.text.tolist(),
    dev.text.tolist(),
    test.text.tolist()
)

ytr = (train.label == "spam").astype(int).to_numpy()
yd = (dev.label == "spam").astype(int).to_numpy()
yt = (test.label == "spam").astype(int).to_numpy()

print("Class counts:")
print(df.label.value_counts())
print("Train:", len(train))
print("Dev:", len(dev))
print("Test:", len(test))


Class counts:
label
ham     4516
spam     642
Name: count, dtype: int64
Train: 3610
Dev: 774
Test: 774
